# Test 1, Part B: Volume, ticket profile, and two operations claims

Part A assessed concepts. Part B assesses whether you can apply them without step-by-step direction: choose a representation, find the main structure in a small operational dataset, then test two claims about that structure and say what the data can and cannot support.

This part is worth 23 points. It opens on publication, Wednesday, September 23, and is due Monday, September 28 at 11:59 p.m. Central for all students. Work individually. Submit one `.ipynb` notebook, as for homework.

**Resources.** Open book, open notes, open documentation. You may use LLM assistance for understanding and debugging under the homework policy: the explanations you submit must be your own, you must be able to explain the work, and you must disclose material assistance at the end.

**Grading rule.** The interpretation prompts must use the numbers your notebook prints. Generic answers that could describe any dataset, and answers whose numbers do not match your outputs, receive no credit. Code earns credit only through the decisions it shows: the representation, the number of components, the statistic, and the null model. Code that runs earns nothing on its own.

**Time.** Designed for about two hours of focused work. No method here is new to the course, and the setup supplies helpers for display and grouping. Parts 3 and 4 carry 12 of the 23 points, so budget your time for them. If you are well past three hours, stop, write down what you have, and submit.

## The case

Rolling Kitchen operates six food trucks in one city. Each truck serves a lunch window and a dinner window on working days. The operations lead has two beliefs about the August 2026 service windows and wants to know whether the recorded data support them:

1. **Dinner windows run at higher volume than lunch windows.** Based on last summer's peak weeks, the lead currently staffs dinner windows for about 40 percent more orders than lunch windows.
2. **Trucks in the north district run at higher volume than trucks in the south district.**

"Volume" is not a single column. Part of your task is to decide how the seven recorded measurements represent it.

**Provenance.** The file covers the six trucks over the twenty working days from August 3 through August 28, 2026, one row per truck per day per window. Nine windows with a recorded point-of-sale outage were removed before the file was prepared. The operator assigned trucks to districts; the documentation does not say how or whether trucks ever move between districts. There is no information about weather, events, menu changes, or staffing.

| Column | Description | Units |
| --- | --- | --- |
| `window_id` | Row label | - |
| `truck` | Truck identifier, T1 to T6 | - |
| `district` | Operating district, `north` or `south` | - |
| `date` | Service date | ISO date |
| `window` | `lunch` or `dinner` | - |
| `orders` | Orders completed in the window | count |
| `items` | Items sold | count |
| `revenue_usd` | Window revenue | US dollars |
| `avg_ticket_usd` | Average order value | US dollars |
| `queue_peak` | Longest observed queue | customers |
| `prep_minutes` | Average preparation time per order | minutes |
| `card_share` | Share of orders paid by card | proportion |

## Setup

The setup loads the file, prints a summary, and defines three helpers you may use anywhere below. Read the summary, but do not spend time on further exploration; the assessed work starts at Part 1.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.decomposition import PCA

DATA_URL = (
    "https://raw.githubusercontent.com/olearydj/INSY7130/"
    "main/tests/test1/data/service-windows.csv"
)
data_candidates = [
    Path("data/service-windows.csv"),
    Path("content/tests/test1/data/service-windows.csv"),
]
data_source = next((path for path in data_candidates if path.is_file()), DATA_URL)
FEATURES = [
    "orders", "items", "revenue_usd", "avg_ticket_usd",
    "queue_peak", "prep_minutes", "card_share",
]
windows = pd.read_csv(data_source).set_index("window_id")
measurements = windows[FEATURES]

print("Rows and columns:", windows.shape)
print(windows["window"].value_counts().to_dict(), windows["district"].value_counts().to_dict())
display(measurements.describe().round(2))
display(measurements.corr().round(2))

The helpers below are supplied so that display and grouping do not cost you time. They contain no decisions.

In [ ]:
def summarize_pca(fitted, feature_names):
    """Return (loadings, variance_summary) tables for a fitted PCA."""
    names = [f"PC{i + 1}" for i in range(fitted.n_components_)]
    loadings = pd.DataFrame(fitted.components_.T, index=feature_names, columns=names)
    variance_summary = pd.DataFrame(
        {
            "Eigenvalue (component variance)": fitted.explained_variance_,
            "Variance retained (%)": 100 * fitted.explained_variance_ratio_,
            "Cumulative retained (%)": 100 * fitted.explained_variance_ratio_.cumsum(),
        },
        index=names,
    )
    return loadings, variance_summary


def plot_scores(scores, labels, x="PC1", y="PC2", xlabel=None, ylabel=None):
    """Scatter two score columns, marking each row by a label Series (for example windows["window"])."""
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for value, marker in zip(sorted(labels.unique()), ("o", "^", "s", "D")):
        mask = labels == value
        ax.scatter(scores.loc[mask, x], scores.loc[mask, y], marker=marker, alpha=0.7, label=str(value))
    ax.axvline(0, color="gray", lw=0.5)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    ax.legend(title=labels.name)
    plt.tight_layout()
    plt.show()


def split_groups(values, labels, first, second):
    """Return (first_array, second_array): the values whose label equals first, then second."""
    return values[labels == first].to_numpy(), values[labels == second].to_numpy()

## Part 1: Represent volume (5 points)

Decide how to represent the seven measurements before reducing them, fit PCA, and identify what its leading components measure. The decisions are yours; the mechanics are not new.

Your work must show:

- **1a.** Whether you standardize the columns, with evidence for the decision. Fit PCA to `measurements` and to a standardized copy, call `summarize_pca` on each, and compare the two PC1 loadings.
- **1b.** Your choice of how many components to keep, with the reason, using the variance summary from your chosen representation.
- **1c.** The loadings of the first two components and a one-line operational name for each, in Rolling Kitchen's terms.
- **1d.** One figure that shows the structure you found. `plot_scores(scores, windows["window"])` is one acceptable choice.

Save the component scores in a DataFrame named `scores` with columns `PC1`, `PC2`, and so on, indexed by `window_id`, and orient PC1 so that a larger score means more volume: if the `orders` loading on PC1 is negative, negate that column of `scores`. Later parts use it.

In [ ]:
# 1a. Standardize, fit PCA to both frames, and compare the PC1 loadings.

In [ ]:
# 1b-1d. Retained variance and your k, the first two loadings, the oriented scores, and the figure.

## Part 2: Test the two claims (6 points)

Use your volume score to test each of the operations lead's claims. `split_groups(values, labels, first, second)` returns the two arrays that a two-sample SciPy call expects.

Your work must show:

- **2a.** A one-number statistic that represents a claim, written as a two-argument function and stated before any test is run, with its sign convention. A signed difference of group means is acceptable; another statistic is acceptable if you say why it represents the claim better.
- **2b.** The null model in one sentence for each claim: what is being shuffled, and what that assumes.
- **2c.** Three SciPy permutation tests, each with 5,000 resamples, a two-sided alternative, and random seed 7130, printing the observed statistic and the simulated p-value for each: claim 1 on your volume score (dinner minus lunch), claim 2 on your volume score (north minus south), and claim 2 on `revenue_usd`. Write a small function or repeat the call, whichever you prefer. Revenue is the figure the operations lead sees in the monthly summary, so the two claim-2 results are what you will reconcile in Part 3.
- **2d.** Two 95 percent bootstrap intervals (5,000 resamples, seed 7130, percentile method), printed: the percent difference in mean `orders`, dinner relative to lunch, which is the quantity behind the lead's staffing figure; and the difference in mean `revenue_usd`, north minus south.

In [ ]:
# 2a-2c. Statistic function and the three permutation results. State each null model in a comment or a short markdown cell.

In [ ]:
# 2d. Bootstrap intervals: percent difference in orders (dinner relative to lunch) and revenue difference (north minus south).

## Part 3: Interpretation (9 points)

Answer each prompt in a short paragraph using the numbers your notebook printed.

**3a. Representation (3 points).** State the first-component loadings from your chosen representation and what PC1 measures. State the retained variance at your chosen number of components. Then say what PC1 measured when the columns were not standardized, and why that would have been the wrong representation of volume.

> Replace this text with your answer to 3a.

**3b. Claim 1 (3 points).** Report the observed volume statistic with its direction and the simulated p-value under your null model. Then report the interval for the percent difference in orders and say whether the 40 percent staffing figure is consistent with it. Give one bounded conclusion the evidence supports and one conclusion it does not support. Do not describe the p-value as the probability that the claim is false.

> Replace this text with your answer to 3b.

**3c. Claim 2 (3 points).** Report the observed statistic and simulated p-value for your volume score and for revenue. Tell the operations lead, in plain language, what the volume result establishes about the belief and why the revenue result differs. Then use the revenue interval to state the range of revenue differences the data are consistent with, and explain why the revenue result's "no evidence of a difference" is not the same as "no difference."

> Replace this text with your answer to 3c.

## Part 4: Bounded claim, reproducibility, and disclosure (3 points)

**4a.** In two or three sentences, state the scope of what this analysis can claim about Rolling Kitchen's operations. Name what the file covers, what was removed, and one thing the provenance leaves unknown that limits the claim.

> Replace this text with your bounded claim for 4a.

Restart the runtime and run all cells in order before submitting. Keep the outputs and check that the numbers in your answers match them.

| Evaluation | Points |
| --- | ---: |
| Part 1: representation decision, k, component naming, figure | 5 |
| Part 2: statistic, null models, three permutation results, two intervals | 6 |
| Part 3: interpretation tied to your outputs | 9 |
| Part 4: bounded claim, reproducibility, and assistance disclosure | 3 |
| **Total** | **23** |

**Assistance disclosure:** Identify material assistance, including any LLM use, and how you checked it. Write `None` if you received no material assistance.

> Replace this text with your disclosure.

**Optional, ungraded feedback:** Approximately how much active time did you spend, excluding breaks? Which part required the most effort, and was anything unclear?

> Optional feedback.